# Q5 Failure Tests — Screenshot Capture Notebook

This notebook runs each of the five Q5 failure scenarios against the live 6-node Docker cluster
and saves a terminal-styled PNG screenshot to `screenshots/` for each test case.
Those PNGs are referenced directly by `report.tex` (`\\includegraphics{screenshots/tcN_*.png}`).

## Prerequisites

1. Docker Desktop is running.
2. The cluster is **not** already up (the notebook manages `make up`/`make down` itself).
3. You are running this notebook from the repo root **or** from `docs/report/`.
   The helper below auto-detects the repo root.
4. Python packages: `matplotlib`, `Pillow` — install with `pip install matplotlib Pillow`.

## Workflow

Run each TC section **in order**, top to bottom.  
Each section:
- Resets the cluster (`make down && make up`)
- Performs the failure action
- Captures log output from the relevant nodes
- Saves a terminal-styled PNG to `docs/report/screenshots/`

After all five sections are done, run `make pdf` from the repo root to build the report.

In [ ]:
"""Shared helpers — run this cell first."""
import os
import subprocess
import time
import textwrap
from pathlib import Path

import matplotlib
matplotlib.use("Agg")          # headless backend — works without a display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Locate repo root ────────────────────────────────────────────────────────
def _find_repo_root() -> Path:
    """Walk upward from this notebook until we find AGENTS.md."""
    here = Path(os.getcwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Cannot find repo root (AGENTS.md not found).")

REPO_ROOT   = _find_repo_root()
COMPOSE_CMD = ["docker", "compose", "-f", str(REPO_ROOT / "server" / "docker-compose.yml")]
SCREENSHOTS = REPO_ROOT / "docs" / "report" / "screenshots"
SCREENSHOTS.mkdir(parents=True, exist_ok=True)

print(f"Repo root : {REPO_ROOT}")
print(f"Screenshots: {SCREENSHOTS}")


# ── Shell helpers ────────────────────────────────────────────────────────────
def run(cmd: list[str], *, check=True, capture=False, timeout=60) -> subprocess.CompletedProcess:
    """Run a command, optionally capturing output."""
    result = subprocess.run(
        cmd,
        capture_output=capture,
        text=True,
        timeout=timeout,
        check=False,
    )
    if check and result.returncode != 0:
        print(f"[WARN] Command {cmd} exited {result.returncode}")
        if result.stderr:
            print(result.stderr[:500])
    return result


def cluster_up(wait: int = 8) -> None:
    """Build and start the 6-node cluster, then wait for it to stabilise."""
    print("Starting cluster…")
    run(COMPOSE_CMD + ["up", "--build", "-d"])
    print(f"Waiting {wait}s for leader election…")
    time.sleep(wait)


def cluster_down() -> None:
    """Tear down the cluster."""
    print("Stopping cluster…")
    run(COMPOSE_CMD + ["down"], check=False)


def docker_logs(service: str, tail: int = 60) -> str:
    """Return the last *tail* log lines from a container."""
    result = run(
        ["docker", "logs", "--tail", str(tail), service],
        capture=True, check=False
    )
    return (result.stdout + result.stderr).strip()


def all_logs(services=("fishing1","fishing2","fishing3","fishing4","fishing5","fishing6"),
             tail: int = 30) -> str:
    """Concatenate recent logs from multiple services with headers."""
    parts = []
    for svc in services:
        lines = docker_logs(svc, tail=tail)
        if lines:
            parts.append(f"=== {svc} ===")
            parts.append(lines)
    return "\n".join(parts)


def find_leader() -> str | None:
    """Grep all node logs for 'became leader' and return the service name."""
    for svc in ["fishing1","fishing2","fishing3","fishing4","fishing5","fishing6"]:
        out = docker_logs(svc, tail=200)
        if "became leader" in out:
            return svc
    return None


# ── Screenshot renderer ──────────────────────────────────────────────────────
def save_terminal_screenshot(
    title: str,
    log_text: str,
    output_path: Path,
    max_lines: int = 55,
) -> None:
    """
    Render *log_text* as a terminal-styled PNG and save to *output_path*.
    Lines are word-wrapped at 110 chars and truncated to *max_lines*.
    """
    # Wrap and truncate
    raw_lines = log_text.splitlines()
    wrapped: list[str] = []
    for line in raw_lines:
        wrapped.extend(textwrap.wrap(line, width=110) or [""])
    if len(wrapped) > max_lines:
        wrapped = [f"[… {len(wrapped) - max_lines} lines omitted …]"] + wrapped[-max_lines:]

    n = len(wrapped)
    fig_h = max(3.0, 0.22 * n + 1.0)
    fig, ax = plt.subplots(figsize=(14, fig_h))
    fig.patch.set_facecolor("#1e1e1e")
    ax.set_facecolor("#1e1e1e")
    ax.axis("off")

    # Title bar
    fig.text(
        0.5, 0.98, title,
        ha="center", va="top",
        fontsize=10, fontweight="bold",
        color="#ffffff",
        fontfamily="monospace",
    )

    # Log body
    body = "\n".join(wrapped)
    ax.text(
        0.01, 0.97, body,
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=7.5,
        fontfamily="monospace",
        color="#d4d4d4",
        linespacing=1.4,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"Saved: {output_path}")


print("Helpers loaded.")

---
## TC1 — Leader Crash and Re-Election

**Goal**: Kill the current leader; verify that a new leader is elected and the client can resume.

**Expected**: Remaining followers detect a missing heartbeat, one starts an election, wins majority, begins heartbeats.

In [ ]:
cluster_down()
cluster_up(wait=10)

leader = find_leader()
print(f"Current leader: {leader}")

In [ ]:
assert leader is not None, "No leader found — check cluster logs with all_logs()"

print(f"Stopping {leader}…")
run(["docker", "stop", leader])

# Wait for re-election (max election timeout = 3s + margin)
print("Waiting 8s for re-election…")
time.sleep(8)

new_leader = find_leader()
print(f"New leader: {new_leader}")

In [ ]:
# Capture logs from all surviving nodes
survivors = [s for s in ["fishing1","fishing2","fishing3","fishing4","fishing5","fishing6"] if s != leader]
logs_tc1 = all_logs(services=survivors, tail=40)
print(logs_tc1[:2000])

In [ ]:
save_terminal_screenshot(
    title=f"TC1: Leader Crash and Re-Election  |  killed={leader}  new_leader={new_leader}",
    log_text=logs_tc1,
    output_path=SCREENSHOTS / "tc1_leader_crash.png",
)

---
## TC2 — Follower Crash and Recovery

**Goal**: Kill a follower, send updates while it is down, restart it, verify log catch-up.

**Expected**: Cluster commits entries with remaining nodes (majority intact); restarted follower receives full log on next heartbeat.

In [ ]:
cluster_down()
cluster_up(wait=10)

leader_tc2 = find_leader()
print(f"Leader: {leader_tc2}")

# Pick a follower (any node that is not the leader)
all_nodes = ["fishing1","fishing2","fishing3","fishing4","fishing5","fishing6"]
follower_tc2 = next(s for s in all_nodes if s != leader_tc2)
print(f"Will stop follower: {follower_tc2}")

In [ ]:
print(f"Stopping {follower_tc2}…")
run(["docker", "stop", follower_tc2])
time.sleep(2)

# The leader should continue committing entries on its own (via other followers)
# We observe this by watching the leader logs for a few heartbeat cycles
print("Waiting 6s (3 heartbeat cycles) while follower is down…")
time.sleep(6)

logs_while_down = docker_logs(leader_tc2 if leader_tc2 else all_nodes[0], tail=30)
print("Leader logs while follower is down:")
print(logs_while_down)

In [ ]:
print(f"Restarting {follower_tc2}…")
run(["docker", "start", follower_tc2])
print("Waiting 5s for catch-up via heartbeat…")
time.sleep(5)

logs_tc2 = all_logs(
    services=[leader_tc2] + [follower_tc2] if leader_tc2 else [follower_tc2],
    tail=40,
)
print(logs_tc2[:2000])

In [ ]:
save_terminal_screenshot(
    title=f"TC2: Follower Crash and Recovery  |  follower={follower_tc2}  leader={leader_tc2}",
    log_text=logs_while_down + "\n\n--- After restart ---\n" + logs_tc2,
    output_path=SCREENSHOTS / "tc2_follower_recovery.png",
)

---
## TC3 — Network Partition (Pause / Unpause)

**Goal**: Pause a container to simulate a network partition, then unpause it.

**Expected**: If the paused node is the leader, re-election occurs among remaining nodes. On unpause the node receives a heartbeat, discovers the current term, and syncs.

In [ ]:
cluster_down()
cluster_up(wait=10)

leader_tc3 = find_leader()
print(f"Leader: {leader_tc3}")

# Pause the leader to get the most interesting behaviour (re-election)
pause_target = leader_tc3 if leader_tc3 else "fishing2"
print(f"Will pause: {pause_target}")

In [ ]:
print(f"Pausing {pause_target}…")
run(["docker", "pause", pause_target])

print("Waiting 8s for re-election and cluster stabilisation…")
time.sleep(8)

survivors_tc3 = [s for s in ["fishing1","fishing2","fishing3","fishing4","fishing5","fishing6"] if s != pause_target]
logs_paused = all_logs(services=survivors_tc3, tail=25)
print(logs_paused[:1500])

In [ ]:
print(f"Unpausing {pause_target}…")
run(["docker", "unpause", pause_target])
print("Waiting 5s for resynchronisation…")
time.sleep(5)

logs_tc3 = all_logs(services=[pause_target] + survivors_tc3[:2], tail=30)
print(logs_tc3[:2000])

In [ ]:
save_terminal_screenshot(
    title=f"TC3: Network Partition (pause/unpause)  |  paused={pause_target}",
    log_text=logs_paused + "\n\n--- After unpause ---\n" + logs_tc3,
    output_path=SCREENSHOTS / "tc3_partition.png",
)

---
## TC4 — New Node Joining the Cluster

**Goal**: Start with only 5 nodes running, build up some log history, then bring the 6th node online.

**Expected**: The new node starts as a follower, receives the full log on the first heartbeat, and applies committed entries.

In [ ]:
cluster_down()

# Start only 5 nodes
print("Starting 5-node cluster (fishing1–fishing5)…")
run(COMPOSE_CMD + ["up", "--build", "-d",
                   "fishing1", "fishing2", "fishing3", "fishing4", "fishing5"])
print("Waiting 10s for leader election…")
time.sleep(10)

leader_tc4 = find_leader()
print(f"Leader with 5 nodes: {leader_tc4}")

In [ ]:
# Let the cluster run a few heartbeat cycles to accumulate log entries
print("Waiting 6s to accumulate heartbeats…")
time.sleep(6)

# Now bring fishing6 online
print("Starting fishing6…")
run(COMPOSE_CMD + ["up", "-d", "fishing6"])
print("Waiting 5s for fishing6 to sync…")
time.sleep(5)

logs_tc4_new = docker_logs("fishing6", tail=40)
print("fishing6 logs after join:")
print(logs_tc4_new)

In [ ]:
logs_tc4 = (
    f"=== fishing6 (new node) ===\n{logs_tc4_new}\n\n"
    f"=== {leader_tc4} (leader) ===\n{docker_logs(leader_tc4 or 'fishing1', tail=30)}"
)

save_terminal_screenshot(
    title=f"TC4: New Node Joining  |  joined=fishing6  leader={leader_tc4}",
    log_text=logs_tc4,
    output_path=SCREENSHOTS / "tc4_new_node.png",
)

---
## TC5 — Split Vote and Election Retry

**Goal**: Kill the leader and one follower simultaneously to increase the probability of a split vote among the remaining four nodes.

**Expected**: Multiple candidates start elections in the same term; if no candidate gets a majority the term ends without a leader; a new term starts with fresh random timeouts; eventually a leader wins.

> **Note:** A genuine split vote is non-deterministic.  If one is not observed after the kill,
> the logs will still show the election-timeout mechanism (different timeouts per node) and
> multiple terms before the final leader is elected — which satisfies the assignment alternative.

In [ ]:
cluster_down()
cluster_up(wait=10)

leader_tc5 = find_leader()
all_tc5 = ["fishing1","fishing2","fishing3","fishing4","fishing5","fishing6"]
followers_tc5 = [s for s in all_tc5 if s != leader_tc5]
kill_follower = followers_tc5[0]  # pick an arbitrary follower

print(f"Leader : {leader_tc5}")
print(f"Will also kill follower: {kill_follower}")

In [ ]:
print(f"Stopping {leader_tc5} and {kill_follower} simultaneously…")
# Use a single docker stop call with two args to make it as simultaneous as possible
run(["docker", "stop", leader_tc5, kill_follower])

print("Waiting 12s for election(s) to complete…")
time.sleep(12)

survivors_tc5 = [s for s in all_tc5 if s not in (leader_tc5, kill_follower)]
new_leader_tc5 = find_leader()
print(f"New leader: {new_leader_tc5}")

In [ ]:
logs_tc5 = all_logs(services=survivors_tc5, tail=45)
print(logs_tc5[:2000])

In [ ]:
save_terminal_screenshot(
    title=(
        f"TC5: Split Vote / Election Retry  |"
        f"  killed={leader_tc5},{kill_follower}  new_leader={new_leader_tc5}"
    ),
    log_text=logs_tc5,
    output_path=SCREENSHOTS / "tc5_split_vote.png",
)

---
## Final: Verify Screenshots and Tear Down

In [ ]:
cluster_down()

print("\nScreenshots generated:")
for p in sorted(SCREENSHOTS.glob("tc*.png")):
    size_kb = p.stat().st_size // 1024
    print(f"  {p.name:35s}  {size_kb} KB")

print("\nAll done. Run `make pdf` from the repo root to build the report.")